In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
import os

import threading

# Function to extract 2nd byte (little endian)
def extract_decimal_2nd_byte_le(value):
    try:
        val = int(value)
        second_byte = (val >> 8) & 0xFF
        return second_byte
    except:
        return pd.NA

def upload_and_process():
    input_path = filedialog.askopenfilename(
        title="Select FaultCode CSV File",
        filetypes=[("CSV Files", "*.csv")]
    )
    if not input_path:
        return

    output_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV Files", "*.csv")],
        title="Save Processed CSV As",
        initialfile="Processed_FaultCode_Output.csv"
    )
    if not output_path:
        return

    spinner = tk.Toplevel(root)
    spinner.title("Processing...")
    spinner.configure(bg="#333333")
    tk.Label(spinner, text="⏳ Please wait, processing your file...", font=("Arial", 11, "bold"), fg="white", bg="#333333").pack(padx=20, pady=20)
    spinner.geometry("300x100")
    spinner.resizable(False, False)
    spinner.grab_set()
    root.update()

    def process():
        try:
            df = pd.read_csv(input_path)

            if 'fault_status_4' not in df.columns:
                spinner.destroy()
                messagebox.showerror("Missing Column", "'fault_status_4' column not found in CSV.")
                return

            df['BMS Fault code'] = df['fault_status_4'].apply(extract_decimal_2nd_byte_le)
            df.to_csv(output_path, index=False)

            spinner.destroy()
            messagebox.showinfo("✅ Success", f"File saved successfully as:\n\n{output_path}")

            os.startfile(os.path.dirname(output_path))

        except Exception as e:
            spinner.destroy()
            messagebox.showerror("❌ Error", f"An error occurred:\n{str(e)}")

    threading.Thread(target=process).start()

# GUI Setup
root = tk.Tk()
root.title("🔧 Fault Code Converter Tool (Enhanced)")
root.geometry("500x300")
root.configure(bg="#222831")
root.resizable(False, False)

# Header Frame
header_frame = tk.Frame(root, bg="#00ADB5")
header_frame.pack(fill="x")

title_label = tk.Label(header_frame, text="⚡ Fault Status 4 ➡️ BMS Fault Code Converter", font=("Arial", 13, "bold"), fg="white", bg="#00ADB5", pady=10)
title_label.pack()

# Main Content Frame
content_frame = tk.Frame(root, bg="#222831")
content_frame.pack(expand=True)

info_label = tk.Label(content_frame, text="Convert 'fault_status_4' column to BMS Fault Code (2nd Byte)", font=("Arial", 11), bg="#222831", fg="white", pady=20)
info_label.pack()

process_btn = tk.Button(content_frame, text="🗂️ Select and Process CSV File", command=upload_and_process,
                        font=("Arial", 11, "bold"), bg="#00ADB5", fg="white", width=30, height=2, relief="raised", bd=3)
process_btn.pack(pady=10)

# footer_label = tk.Label(root, text="Developed by Sanmathi | © 2025", font=("Arial", 9), bg="#222831", fg="#AAAAAA")
# footer_label.pack(side="bottom", pady=10)

root.mainloop()